# Extended Thinking with Tool Use

## Table of contents
- [Setup](#setup)
- [Basic example](#basic-example)
- [Multiple tool calls](#multiple-tool-calls-with-thinking)
- [Preserving thinking blocks](#preserving-thinking-blocks)

This notebook demonstrates how to use Claude 3.7 Sonnet's extended thinking feature with tools. The extended thinking feature allows you to see Claude's step-by-step thinking before it provides a final answer, providing transparency into how it decides which tools to use and how it interprets tool results.

When using extended thinking with tool use, the model will show its thinking before making tool requests, but not repeat the thinking process after receiving tool results. Claude will not output another thinking block until after the next non-`tool_result` `user` turn. For more information on extended thinking, see our [documentation](https://docs.claude.com/en/docs/build-with-claude/extended-thinking).

## Setup

First, let's install the necessary packages and set up our environment.

In [ ]:
// No pip install needed. Ensure dependencies are installed:
// npm install @anthropic-ai/sdk


In [ ]:
import Anthropic from '@anthropic-ai/sdk';

const MODEL_NAME = 'claude-sonnet-4-6';
const MAX_TOKENS = 4000;
const THINKING_BUDGET_TOKENS = 2000;

const client = new Anthropic();

function printThinkingResponse(response: Anthropic.Message): void {
  console.log('\n==== FULL RESPONSE ====');
  for (const block of response.content) {
    if (block.type === 'thinking') {
      const t = (block as any).thinking || '';
      console.log('\nTHINKING BLOCK:');
      console.log(t.length > 500 ? t.slice(0, 500) + '...' : t);
    } else if (block.type === 'text') {
      console.log('\nFINAL ANSWER:');
      console.log(block.text);
    }
  }
  console.log('\n==== END RESPONSE ====');
}

async function countTokens(messages: Anthropic.MessageParam[], tools?: Anthropic.Tool[]): Promise<number> {
  const params: any = { model: MODEL_NAME, messages };
  if (tools) params.tools = tools;
  const result = await client.messages.countTokens(params);
  return result.input_tokens;
}


## Single tool calls with thinking

This example demonstrates how to combine thinking and make a single tool call, with a mock weather tool.

In [ ]:
async function toolUseWithThinking(): Promise<void> {
  const tools: Anthropic.Tool[] = [{
    name: 'weather',
    description: 'Get current weather information for a location.',
    input_schema: {
      type: 'object',
      properties: { location: { type: 'string', description: 'The location to get weather for.' } },
      required: ['location']
    }
  }];

  const weatherData: Record<string, { temperature: number; condition: string }> = {
    'New York': { temperature: 72, condition: 'Sunny' },
    'London': { temperature: 62, condition: 'Cloudy' },
    'Tokyo': { temperature: 80, condition: 'Partly cloudy' },
    'Paris': { temperature: 65, condition: 'Rainy' },
    'Sydney': { temperature: 85, condition: 'Clear' },
    'Berlin': { temperature: 60, condition: 'Foggy' },
  };

  function weather(location: string) {
    return weatherData[location] ?? { error: 'No weather data available for ' + location };
  }

  let response = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: MAX_TOKENS,
    thinking: { type: 'enabled', budget_tokens: THINKING_BUDGET_TOKENS },
    tools,
    messages: [{ role: 'user', content: "What's the weather like in Paris today?" }]
  });

  const assistantBlocks = response.content.filter(b =>
    b.type === 'thinking' || b.type === 'redacted_thinking' || b.type === 'tool_use'
  );

  const fullConversation: Anthropic.MessageParam[] = [{ role: 'user', content: "What's the weather like in Paris today?" }];

  if (response.stop_reason === 'tool_use') {
    fullConversation.push({ role: 'assistant', content: assistantBlocks as any });

    const toolUseBlock = response.content.find(b => b.type === 'tool_use') as Anthropic.ToolUseBlock | undefined;
    if (toolUseBlock) {
      const location = (toolUseBlock.input as any).location as string;
      const toolResult = weather(location);
      fullConversation.push({
        role: 'user',
        content: [{ type: 'tool_result', tool_use_id: toolUseBlock.id, content: JSON.stringify(toolResult) }]
      });

      response = await client.messages.create({
        model: MODEL_NAME,
        max_tokens: MAX_TOKENS,
        thinking: { type: 'enabled', budget_tokens: THINKING_BUDGET_TOKENS },
        tools,
        messages: fullConversation
      });
    }
  }

  printThinkingResponse(response);
}

await toolUseWithThinking();


## Multiple tool calls with thinking

This example demonstrates how to handle multiple tool calls, such as a mock news and weather service, while observing the thinking process.

In [ ]:
async function multipleToolCallsWithThinking(): Promise<void> {
  const tools: Anthropic.Tool[] = [
    {
      name: 'weather',
      description: 'Get current weather information for a location.',
      input_schema: { type: 'object', properties: { location: { type: 'string', description: 'The location to get weather for.' } }, required: ['location'] }
    },
    {
      name: 'news',
      description: 'Get latest news headlines for a topic.',
      input_schema: { type: 'object', properties: { topic: { type: 'string', description: 'The topic to get news about.' } }, required: ['topic'] }
    }
  ];

  function weather(location: string) {
    const weatherData: Record<string, { temperature: number; condition: string }> = {
      'New York': { temperature: 72, condition: 'Sunny' },
      'London': { temperature: 62, condition: 'Cloudy' },
      'Tokyo': { temperature: 80, condition: 'Partly cloudy' },
      'Paris': { temperature: 65, condition: 'Rainy' },
      'Sydney': { temperature: 85, condition: 'Clear' },
      'Berlin': { temperature: 60, condition: 'Foggy' },
    };
    return weatherData[location] ?? { error: 'No weather data available for ' + location };
  }

  function news(topic: string) {
    const newsData: Record<string, string[]> = {
      technology: ['New AI breakthrough announced by research lab', 'Tech company releases latest smartphone model'],
      sports: ['Local team wins championship game'],
      weather: ['Storm system developing in the Atlantic']
    };
    return { headlines: newsData[topic.toLowerCase()] ?? ['No news available for this topic'] };
  }

  let response = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: MAX_TOKENS,
    thinking: { type: 'enabled', budget_tokens: THINKING_BUDGET_TOKENS },
    tools,
    messages: [{ role: 'user', content: "What's the weather in London, and can you also tell me the latest news about technology?" }]
  });

  const fullConversation: Anthropic.MessageParam[] = [{ role: 'user', content: "What's the weather in London, and can you also tell me the latest news about technology?" }];

  while (response.stop_reason === 'tool_use') {
    const assistantBlocks = response.content.filter(b => b.type === 'thinking' || b.type === 'redacted_thinking' || b.type === 'tool_use');
    fullConversation.push({ role: 'assistant', content: assistantBlocks as any });

    const toolUseBlock = response.content.find(b => b.type === 'tool_use') as Anthropic.ToolUseBlock | undefined;
    if (!toolUseBlock) break;

    let toolResult: any;
    if (toolUseBlock.name === 'weather') toolResult = weather((toolUseBlock.input as any).location);
    else if (toolUseBlock.name === 'news') toolResult = news((toolUseBlock.input as any).topic);
    else toolResult = { error: 'Unknown tool' };

    fullConversation.push({ role: 'user', content: [{ type: 'tool_result', tool_use_id: toolUseBlock.id, content: JSON.stringify(toolResult) }] });

    response = await client.messages.create({
      model: MODEL_NAME,
      max_tokens: MAX_TOKENS,
      thinking: { type: 'enabled', budget_tokens: THINKING_BUDGET_TOKENS },
      tools,
      messages: fullConversation
    });
  }

  printThinkingResponse(response);
}

await multipleToolCallsWithThinking();


## Preserving thinking blocks

When working with extended thinking and tools, make sure to:

1. **Preserve thinking block signatures**: Each thinking block contains a cryptographic signature that validates the conversation context. These signatures must be included when passing thinking blocks back to Claude.

2. **Avoid modifying prior context**: The API will reject requests if any previous content (including thinking blocks) is modified when submitting a new request with tool results.

3. **Handle both thinking and redacted_thinking blocks**: Both types of blocks must be preserved in the conversation history, even if the content of redacted blocks is not human readable.

For more details on extended thinking without tools, see the main "Extended Thinking" notebook.

In [ ]:
async function thinkingBlockPreservationExample(): Promise<void> {
  const tools: Anthropic.Tool[] = [{
    name: 'weather',
    description: 'Get current weather information for a location.',
    input_schema: { type: 'object', properties: { location: { type: 'string', description: 'The location to get weather for.' } }, required: ['location'] }
  }];

  const weatherData: Record<string, { temperature: number; condition: string }> = {
    'New York': { temperature: 72, condition: 'Sunny' },
    'London': { temperature: 62, condition: 'Cloudy' },
    'Tokyo': { temperature: 80, condition: 'Partly cloudy' },
    'Paris': { temperature: 65, condition: 'Rainy' },
    'Sydney': { temperature: 85, condition: 'Clear' },
    'Berlin': { temperature: 60, condition: 'Foggy' },
  };

  function weather(location: string) {
    return weatherData[location] ?? { error: 'No weather data available for ' + location };
  }

  const response = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: MAX_TOKENS,
    thinking: { type: 'enabled', budget_tokens: THINKING_BUDGET_TOKENS },
    tools,
    messages: [{ role: 'user', content: "What's the weather like in Berlin right now?" }]
  });

  const thinkingBlocks = response.content.filter(b => b.type === 'thinking');
  const toolUseBlocks = response.content.filter(b => b.type === 'tool_use') as Anthropic.ToolUseBlock[];

  if (toolUseBlocks.length > 0) {
    const toolBlock = toolUseBlocks[0];
    const toolResult = weather((toolBlock.input as any).location);

    try {
      await client.messages.create({
        model: MODEL_NAME,
        max_tokens: MAX_TOKENS,
        thinking: { type: 'enabled', budget_tokens: THINKING_BUDGET_TOKENS },
        tools,
        messages: [
          { role: 'user', content: "What's the weather like in Berlin right now?" },
          { role: 'assistant', content: toolUseBlocks as any },
          { role: 'user', content: [{ type: 'tool_result', tool_use_id: toolBlock.id, content: JSON.stringify(toolResult) }] }
        ]
      });
    } catch (e) {
      console.log('Expected error when thinking blocks are omitted: ' + e);
    }

    const completeBlocks = [...thinkingBlocks, ...toolUseBlocks];
    const completeResponse = await client.messages.create({
      model: MODEL_NAME,
      max_tokens: MAX_TOKENS,
      thinking: { type: 'enabled', budget_tokens: THINKING_BUDGET_TOKENS },
      tools,
      messages: [
        { role: 'user', content: "What's the weather like in Berlin right now?" },
        { role: 'assistant', content: completeBlocks as any },
        { role: 'user', content: [{ type: 'tool_result', tool_use_id: toolBlock.id, content: JSON.stringify(toolResult) }] }
      ]
    });

    printThinkingResponse(completeResponse);
  }
}

await thinkingBlockPreservationExample();


## Conclusion

This notebook shows how to combine Claude's extended thinking feature with tool use. Key benefits include:

1. Transparency into Claude's thinking process when using tools
2. Visibility into how Claude decides when to use tools versus internal knowledge
3. Better understanding of multi-step tasks that involve multiple tool calls
4. Insight into how Claude interprets tool results and incorporates them into responses

When using extended thinking with tools, keep in mind:
- Set appropriate thinking budgets for complex tasks (minimum 1,024 tokens)
- Always preserve thinking blocks and their signatures when passing tool results
- Include both normal and redacted thinking blocks in the conversation history
- Ensure that system prompts, tools, and thinking configurations match between calls
- Expect that tool result turns will not contain additional thinking blocks
- Tool use and thinking together can increase token usage and response time